# MLP 構成比較実験

MNIST データセットを使って、MLP の構成の違いが学習にどう影響するかを実験します。

## 実験一覧
1. **層の深さ**: 2層 vs 3層 vs 5層
2. **活性化関数**: ReLU vs Sigmoid vs Tanh
3. **次元の減らし方**: 急に減らす vs 緩やかに vs 同じ幅を維持

## セットアップ

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# デバイス設定
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"使用デバイス: {device}")

In [ ]:
# 共通ハイパーパラメータ
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
EPOCHS = 10

# データ準備
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"訓練データ: {len(train_dataset)} 枚 / テストデータ: {len(test_dataset)} 枚")

## 共通の学習・評価関数

全実験で使い回すため、学習と評価を関数にまとめます。

In [ ]:
def train_model(model, train_loader, epochs=EPOCHS, lr=LEARNING_RATE):
    """モデルを学習し、エポックごとの loss を返す"""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    losses = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        losses.append(avg_loss)
        print(f"  Epoch [{epoch+1}/{epochs}]  Loss: {avg_loss:.4f}")

    return losses


def evaluate_model(model, test_loader):
    """テストデータで正解率を計算して返す"""
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    return accuracy


def count_params(model):
    """モデルの総パラメータ数を返す"""
    return sum(p.numel() for p in model.parameters())


def plot_comparison(results, title):
    """学習曲線と正解率を並べて表示する"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # 学習曲線
    for name, data in results.items():
        ax1.plot(range(1, EPOCHS + 1), data["losses"], marker="o", label=name)
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title(f"{title} — 学習曲線")
    ax1.legend()
    ax1.grid(True)

    # 正解率バーチャート
    names = list(results.keys())
    accs = [results[n]["accuracy"] for n in names]
    params = [results[n]["params"] for n in names]
    bars = ax2.bar(names, accs, color=["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"][:len(names)])
    ax2.set_ylabel("正解率 (%)")
    ax2.set_title(f"{title} — テスト正解率")
    ax2.set_ylim(max(0, min(accs) - 5), 100)
    # バーの上にパラメータ数と正解率を表示
    for bar, acc, p in zip(bars, accs, params):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 f"{acc:.1f}%\n({p:,} params)", ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.show()


print("共通関数を定義しました。")

---
## 実験1: 層の深さ比較

活性化関数は全て ReLU に統一し、層の数だけを変えて比較します。

| モデル | 構成 |
|--------|------|
| 2層 | 784 → 256 → 10 |
| 3層 | 784 → 256 → 128 → 10 |
| 5層 | 784 → 512 → 256 → 128 → 64 → 10 |

In [ ]:
class MLP_2Layer(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 10),
        )
    def forward(self, x):
        return self.net(x)


class MLP_3Layer(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.net(x)


class MLP_5Layer(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64),  nn.ReLU(),
            nn.Linear(64, 10),
        )
    def forward(self, x):
        return self.net(x)


depth_models = {
    "2層 (784→256→10)": MLP_2Layer,
    "3層 (784→256→128→10)": MLP_3Layer,
    "5層 (784→512→...→10)": MLP_5Layer,
}

print("実験1: 層の深さ比較")
depth_results = {}
for name, ModelClass in depth_models.items():
    print(f"\n--- {name} ---")
    model = ModelClass()
    p = count_params(model)
    print(f"  パラメータ数: {p:,}")
    losses = train_model(model, train_loader)
    acc = evaluate_model(model, test_loader)
    print(f"  テスト正解率: {acc:.2f}%")
    depth_results[name] = {"losses": losses, "accuracy": acc, "params": p}

In [ ]:
plot_comparison(depth_results, "実験1: 層の深さ")

---
## 実験2: 活性化関数比較

3層構成 (784 → 256 → 128 → 10) で統一し、活性化関数だけを変えて比較します。

| 活性化関数 | 微分の最大値 | 特徴 |
|-----------|------------|------|
| ReLU | 1 (x>0) | 勾配消失しにくい |
| Sigmoid | 0.25 | 勾配消失しやすい |
| Tanh | 1 | Sigmoid より範囲が広い (-1〜1) |

In [ ]:
class MLP_Activation(nn.Module):
    def __init__(self, activation):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), activation(),
            nn.Linear(256, 128), activation(),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.net(x)


activation_models = {
    "ReLU": nn.ReLU,
    "Sigmoid": nn.Sigmoid,
    "Tanh": nn.Tanh,
}

print("実験2: 活性化関数比較")
activation_results = {}
for name, act in activation_models.items():
    print(f"\n--- {name} ---")
    model = MLP_Activation(act)
    p = count_params(model)
    print(f"  パラメータ数: {p:,}")
    losses = train_model(model, train_loader)
    acc = evaluate_model(model, test_loader)
    print(f"  テスト正解率: {acc:.2f}%")
    activation_results[name] = {"losses": losses, "accuracy": acc, "params": p}

In [ ]:
plot_comparison(activation_results, "実験2: 活性化関数")

---
## 実験2b: 活性化関数比較（10層）

3層では勾配消失の差が見えなかった。σ' の掛け算回数を増やすため、**10層 (σ' が9回掛かる)** で再実験する。

全層を幅 256 で統一し、活性化関数だけを変える。

```
784 → 256 → 256 → 256 → 256 → 256 → 256 → 256 → 256 → 256 → 10
      ~~~   ~~~   ~~~   ~~~   ~~~   ~~~   ~~~   ~~~   ~~~
      活性化関数が9回入る → σ' が9回掛け算される
```

In [ ]:
class MLP_Deep(nn.Module):
    def __init__(self, activation):
        super().__init__()
        layers = [nn.Flatten(), nn.Linear(784, 256), activation()]
        for _ in range(8):  # 隠れ層を8つ追加 → 合計10層
            layers += [nn.Linear(256, 256), activation()]
        layers.append(nn.Linear(256, 10))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


deep_activation_models = {
    "ReLU (10層)": nn.ReLU,
    "Sigmoid (10層)": nn.Sigmoid,
    "Tanh (10層)": nn.Tanh,
}

print("実験2b: 活性化関数比較（10層）")
deep_activation_results = {}
for name, act in deep_activation_models.items():
    print(f"\n--- {name} ---")
    model = MLP_Deep(act)
    p = count_params(model)
    print(f"  パラメータ数: {p:,}")
    losses = train_model(model, train_loader)
    acc = evaluate_model(model, test_loader)
    print(f"  テスト正解率: {acc:.2f}%")
    deep_activation_results[name] = {"losses": losses, "accuracy": acc, "params": p}

In [ ]:
plot_comparison(deep_activation_results, "実験2b: 活性化関数（10層）")

---
## 実験3: 次元の減らし方比較

活性化関数は全て ReLU で統一し、隠れ層の次元の減らし方を変えて比較します。

| パターン | 構成 |
|---------|------|
| 急に減らす | 784 → 64 → 10 |
| 緩やかに減らす | 784 → 256 → 128 → 10 |
| 同じ幅を維持 | 784 → 256 → 256 → 10 |

In [ ]:
class MLP_Steep(nn.Module):
    """急に減らす: 784 → 64 → 10"""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 64), nn.ReLU(),
            nn.Linear(64, 10),
        )
    def forward(self, x):
        return self.net(x)


class MLP_Gradual(nn.Module):
    """緩やかに減らす: 784 → 256 → 128 → 10"""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.net(x)


class MLP_SameWidth(nn.Module):
    """同じ幅を維持: 784 → 256 → 256 → 10"""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 10),
        )
    def forward(self, x):
        return self.net(x)


dim_models = {
    "急に減らす (784→64→10)": MLP_Steep,
    "緩やかに (784→256→128→10)": MLP_Gradual,
    "同じ幅 (784→256→256→10)": MLP_SameWidth,
}

print("実験3: 次元の減らし方比較")
dim_results = {}
for name, ModelClass in dim_models.items():
    print(f"\n--- {name} ---")
    model = ModelClass()
    p = count_params(model)
    print(f"  パラメータ数: {p:,}")
    losses = train_model(model, train_loader)
    acc = evaluate_model(model, test_loader)
    print(f"  テスト正解率: {acc:.2f}%")
    dim_results[name] = {"losses": losses, "accuracy": acc, "params": p}

In [ ]:
plot_comparison(dim_results, "実験3: 次元の減らし方")

---
## 全実験サマリー

In [ ]:
print("=" * 60)
print("全実験サマリー")
print("=" * 60)

all_experiments = [
    ("実験1: 層の深さ", depth_results),
    ("実験2: 活性化関数", activation_results),
    ("実験3: 次元の減らし方", dim_results),
]

for exp_name, results in all_experiments:
    print(f"\n{exp_name}")
    print("-" * 50)
    for name, data in results.items():
        print(f"  {name:<30s}  正解率: {data['accuracy']:.2f}%  パラメータ: {data['params']:>10,}")

    best = max(results, key=lambda k: results[k]["accuracy"])
    print(f"  → ベスト: {best}")